In [1]:
import os
import json
import pickle
import random
import numpy as np

# Path and file searching

In [2]:
# os.getcwd()

In [3]:
path_to_json_dataset_folder = "oct8"

In [4]:
# find all related files first
raw_file_list = []

for root, dirs, files in os.walk(path_to_json_dataset_folder):
    for file in files:
        if file.lower().endswith('.json'):
            # only allowed formula types, no "negation" and "or"
            if  "type0000" in file.lower() or \
                "type0001" in file.lower() or \
                "type0002" in file.lower() or \
                "type0004" in file.lower() or \
                "type0005" in file.lower() or \
                "type0008" in file.lower() or \
                "type0009" in file.lower() :
                
                # remove cache files from our list
                if ".ipynb_checkpoints" not in os.path.join(root, file):
#                     print(os.path.join(root, file))
                    raw_file_list.append(os.path.join(root, file))

In [5]:
# aggregation files into groups based on formula types
file_groups = {}
for t in ["type0000",
          "type0001",
          "type0002",
          "type0004",
          "type0005",
          "type0008",
          "type0009"]:
    for file_path in raw_file_list:
        if t in file_path:
            if t not in file_groups:
                file_groups[t] = [file_path]
            else:
                file_groups[t].append(file_path)
        

In [6]:
file_groups

{'type0000': ['oct8/test_type0000_soft_efo1_qaa.json'],
 'type0001': ['oct8/test_type0001_soft_efo1_qaa.json'],
 'type0002': ['oct8/test_type0002_soft_efo1_qaa.json'],
 'type0004': ['oct8/test_type0004_soft_efo1_qaa.json'],
 'type0008': ['oct8/test_type0008_soft_efo1_qaa.json']}

# Helper Functions

In [7]:
# find 4 unique f1 candidates, one with the max f1 value 
def should_include(f1_values):
    # return the first element for each unique value
    uniques, indices = np.unique(np.array(f1_values), return_index=True)
    
    # cannot distinguish max
    if len(uniques) < 2:
        return []
    elif len(indices) == 4:
        return indices
    elif len(indices) > 4:
        return indices[-4:len(indices)]
    else:
        left = 4 - len(uniques)
        for i in range(indices[0]+1,indices[-1]):
            if left > 0 and i not in indices:
                indices = np.append(indices,i)
                left = left - 1
        # not sufficient
        if left != 0:
            return []
        
        return indices

# Load json

In [8]:
for t in ["type0000",
          "type0001",
          "type0002",
          "type0004",
          "type0005",
          "type0008",
          "type0009"]:
    
    if t not in file_groups:
        continue
        
    curr_data_list = file_groups[t]

    idx = 0
    dataset = []

    for f in curr_data_list:
        raw_data = json.load(open(f))
        
        for query in raw_data:
            formula = list(query.keys())[0]
            
            # one max f1_value + 3 lower values
            indices = should_include(query[formula][2]["f1_values"])

            # no enough values for chatgpt  
            if len(indices) != 4:
                continue

            # format: (id, origin_string_query_type, dictionary , [4 candidate], [4 f1_values], correct_anwerer(max_candi))
            selected_f1_answes = np.array(query[formula][1]['f1_answers'])[indices]
            selected_f1_values = np.array(query[formula][2]['f1_values'])[indices]
            max_candidate = selected_f1_answes[np.argmax(selected_f1_values)]

            row = (idx, formula, query[formula][0], selected_f1_answes, selected_f1_values, max_candidate)
            dataset.append(row)
            idx = idx + 1
    pickle_file = open("oct8/{}.pickle".format(t), "wb")
    pickle.dump(dataset, pickle_file)
    pickle_file.close()

In [19]:
test = open("oct8/type0000.pickle","rb")
pickle.load(test)[:5]

[(0,
  'r1(s1,f1,75%,0.5)',
  {'r1': 0, 's1': 1830},
  array([ 2417,  4174, 12585,  2592]),
  array([0.4494, 0.4858, 0.4882, 0.5   ]),
  2592),
 (1,
  'r1(s1,f1,25%,0.3)',
  {'r1': 0, 's1': 2800},
  array([ 860, 4115,  649,  288]),
  array([0.2099, 0.2128, 0.2294, 0.2564]),
  288),
 (2,
  'r1(s1,f1,25%,0.7)',
  {'r1': 0, 's1': 1484},
  array([ 912, 9146, 9447, 6759]),
  array([0.6244, 0.66  , 0.6712, 0.7   ]),
  6759),
 (3,
  'r1(s1,f1,75%,0.8)',
  {'r1': 0, 's1': 4703},
  array([  236,  5997, 11798, 11634]),
  array([0.5674, 0.5795, 0.5674, 0.5674]),
  5997),
 (4,
  'r1(s1,f1,75%,0.5)',
  {'r1': 0, 's1': 1685},
  array([  293,  5250, 12139, 11621]),
  array([0.3547, 0.3908, 0.3547, 0.3547]),
  5250)]